# ICS 604: APPLIED DATA SCIENCE

## Clustering

- ### Introduction
- ### Distance (or Similarity) Measures
- ### Hierarchical Clustering
- ### K-Means Clustering
- ### Evaluating Clustering Results

---

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import string

In [ ]:
np.random.seed(0)

## What is Clustering

Clustering analysis is a computational approach used to uncover structure in a dataset by identifying natural groupings of data points. The goal is to find subsets of points that are “lumped” together based on some notion of similarity. Rather than relying on predefined categories, clustering allows patterns to emerge directly from the data itself.

Clustering is a type of machine learning task, specifically belonging to the class of **unsupervised learning**. In this setting, the analyst does not provide labeled examples or specify which points should belong together. This contrasts with supervised learning, where models are trained using labeled data — for example, distinguishing between spam and legitimate emails. In clustering, the structure must be inferred solely from the relationships within the data.

### What is a Cluster?

A cluster is often informally described as a group of points that are similar or close to each other. However, this definition is not precise, and the concept of a cluster can be somewhat subjective. The idea of where one cluster ends and another begins is not always clearly defined, and different interpretations may lead to different groupings.<br>
  
<center><img src="https://www.dropbox.com/scl/fi/51c1r54rmvb1ww77eukh5/cluster_def.png?rlkey=6me8b45pwc002cp9qu4cv3jxt&st=xtw0az6c&dl=1" width="400"/></center>
<center>Data clustering: 50 years beyond K-means, Jain 2009</center><br>

In practice, we often recognize clusters intuitively when visualizing data, even if it is difficult to formalize the definition. For example, it is not always clear whether a small number of very similar points should be considered a cluster, or whether a cluster requires a larger, more structured grouping.

A more rigorous way to think about clusters is in terms of data density. A cluster can be viewed as a contiguous region in the data space where points are densely packed together. Within such a region, points are relatively close and similar to one another, and the region contains more than just a few points that happen to coincide randomly.

These dense regions are typically separated by areas of lower point density, which helps distinguish one cluster from another. While points in different clusters are generally dissimilar on average, the separation is not always perfect. Some points near the boundaries may be closer to points in another cluster than to those within their own, especially when clusters are not well-defined or overlap slightly.
  
<center><img src="https://www.dropbox.com/scl/fi/2z6t90808lamnszpeawux/two_well_defined_clusters.png?rlkey=fmy7v850bo5ee0xriqgkyt1g8&st=rrvwzvba&dl=1" alt="drawing" width="300"> <img src="https://www.dropbox.com/scl/fi/s2xbbyukwzaqyxdtv7jtt/two_cluster_less_well_defined.png?rlkey=15qeqdrofrlk9om1pdsgqj6yg&st=s39zukcs&dl=1" alt="drawing" width="300"/></center><br>

Overall, clustering involves balancing intuition with formal definitions, as the notion of a “cluster” depends on both the structure of the data and the method used to identify it.

### Data Shapes

In practice, clusters are not always simple or well-behaved. While many introductory examples show clusters as compact, round, and clearly separated groups, real-world data often contains clusters with complex and irregular shapes. These clusters may be elongated, curved, or intertwined, making them much harder to identify using simple assumptions about distance or symmetry.

As a result, some clusters do not neatly fit the earlier definition of compact, high-density regions. Instead, they may form intricate structures that still represent meaningful groupings but are not easily captured by basic clustering methods. For example, datasets like the “two moons” pattern illustrate how clusters can follow curved paths and remain distinct despite being close in certain areas.<br>

<center><img src="https://www.dropbox.com/scl/fi/g6isjoc861s77974ygg2b/data_moons.png?rlkey=9xk7xdzn8od74nabj0htra6ff&st=xbd5v4i5&dl=1" alt="drawing" width="400"/></center><br>

These more complicated shapes can pose challenges for simple clustering approaches, particularly those that assume clusters are spherical or evenly distributed. Consequently, more advanced methods are often required to correctly identify and separate such non-standard cluster structures.

### Applications of Clustering

Clustering is a versatile technique that can be applied across a wide range of domains, including genetics, genomics, user behavior analysis, and finance. Despite the diversity of applications, the underlying idea remains consistent: clusters represent regions of high data density where points within the same region are highly similar, while points in different regions are relatively dissimilar. What varies from one domain to another is how similarity or distance is defined. For example, similarity between two users on a streaming platform like Netflix might be based on how similarly they rate movies or even just the overlap in what they have watched. In finance, similarity between stocks — such as Apple and Qualcomm — might be defined based on how their prices move together, which can be useful for constructing diversified investment portfolios.

One important application of clustering is **data reconciliation**, where the goal is to identify different representations of the same entity. This is particularly valuable in industries that deal with messy or inconsistent data. For instance, variations like “First Avenue,” “First Ave,” “1st Ave,” or even misspelled versions may all refer to the same location. Clustering helps group these variations together, enabling more accurate data integration and analysis.<br>

<center><img src="https://www.dropbox.com/scl/fi/vgaf00vh72gap3trjcebk/clustering_strings.png?rlkey=zyl84x16hdr869niues6attfg&st=llzhnv4m&dl=1" alt="drawing" width="450"></center><br>

Another application is in **financial analysis**, particularly for clustering stocks. By grouping together stocks that behave similarly, analysts can better understand relationships within the market and design strategies for diversification. Instead of selecting assets arbitrarily, clustering provides a data-driven way to identify groups of correlated securities and reduce risk by spreading investments across different clusters.<br>

<center><img src="https://www.dropbox.com/scl/fi/7mwieiqqriwpqaico2ep4/stocks_clusters.png?rlkey=rm57oq0b3la6kdm4sls8ix09k&st=3hbwqsa2&dl=1" alt="drawing" width="600"/></center><br>

Clustering is also widely used in **market segmentation**, where businesses aim to identify distinct groups of customers. By analyzing purchasing behavior, preferences, or usage patterns, companies can discover natural segments within their customer base. This helps answer questions such as who is buying a product, what features are most valued, and how to tailor offerings to different groups. As a result, clustering plays a key role in improving marketing strategies, product design, and customer experience.<br>

<center><img src="https://www.dropbox.com/scl/fi/b13wkc6bewnt13gdc4jau/market-segmentation.png?rlkey=shcxw77yd26rf3pzc4rbjxxha&st=ien3jzxe&dl=1" alt="drawing" width="400"/>

## Distance and Similarity Measures 
  
Clustering fundamentally relies on the ability to quantify how similar or dissimilar two data points are. This is typically done using a **distance function $d(x,y)$**, which takes two points and returns a numerical value representing how different they are. Larger values indicate greater dissimilarity, while smaller values indicate that the points are more alike. In some cases, it is more natural to define a **similarity function $s(x,y)$**, which measures how alike two points are instead. These two concepts are closely related, and a similarity measure can often be converted into a distance measure and vice versa.

Although data points do not always naturally lie in a geometric space, it is often useful to think of them as if they do. By representing data as vectors in a (possibly high-dimensional) space, we can apply well-understood geometric distance measures and leverage efficient algorithms designed for such settings. This abstraction becomes especially powerful when working with complex datasets, as it allows us to use mathematical tools from linear algebra and geometry.

In some cases, defining distance is straightforward. For example, when points are embedded in a geometric space, we can use familiar measures such as Euclidean distance. However, many real-world datasets exist in high-dimensional spaces, where each dimension represents a feature, and the notion of distance becomes less intuitive but still computationally tractable.<br>
  
<center><img src="https://www.dropbox.com/scl/fi/8n2k7phzntz6zqzd5q3wx/distance_a_b.png?rlkey=xm4tgxc2s6xyu6fgbek01rl64&st=e0g8xne1&dl=1" alt="drawing" style="width:250px"/></center><br>
    
In other cases, defining a meaningful distance is much less obvious. For instance, when clustering music genres, it is not immediately clear how to measure the “distance” between two songs. To address this, we first convert each song into a set of features, such as tempo (beats per minute), instrumentation, lyrical characteristics, or the presence of certain words. These features form a vector representation of each song, allowing us to treat them as points in a high-dimensional space. Once this representation is established, standard distance measures (e.g., Euclidean distance) can be used to quantify similarity and perform clustering.

<center><img src="https://www.dropbox.com/scl/fi/mzl1zzb6mayz9cfslz05g/music_genres.png?rlkey=frh769tp5bwgm1surs53it93p&st=xid5nwwb&dl=1" width=550/></center>

<center>Genre Complexes in Popular Music. Silver et al., Plos One, 2016</center>

### Similarity to Distance

In many applications, similarity scores are naturally bounded within a fixed range, often between 0 and 1, where 1 indicates identical items and 0 indicates complete dissimilarity. These normalized similarity measures can be easily converted into distances. A common approach is to define distance as:

$$ d(x,y)=1−s(x,y), $$

which ensures that higher similarity corresponds to smaller distance. This simple transformation allows us to switch between similarity-based and distance-based perspectives depending on the requirements of the clustering algorithm.<br>

<center><img src="https://www.dropbox.com/scl/fi/tzy4rofi1w51551wd3old/dist_sim.png?rlkey=gtbyb8yn7hh09ckmp13lp4uao&st=9ir7mzyn&dl=1" alt="drawing" width="400"/></center><br>

Below are examples of other ways to convert similarity to a distance.


In [ ]:
similarity = np.arange(0, 1.01, 0.01)

def sim_plot(sim, dist, title):
    plt.figure(figsize=(5, 2))
    plt.plot(sim, dist)
    plt.xlabel("Similarity", fontsize=12)
    plt.ylabel("Distance", fontsize=12)
    plt.title(title)
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)
    

method_1 = 1 - similarity
sim_plot(similarity, method_1, "distance = 1 - similarity")

method_2 = -np.log(similarity + 0.0001)
sim_plot(similarity, method_2, "distance = -log(similarity + 0.0001")

method_3 = np.sqrt(1 - similarity)
sim_plot(similarity, method_3, "distance = sqrt(1 - similarity)")


Ultimately, the choice of distance or similarity measure is highly domain-specific. A well-designed measure captures the true notion of similarity relevant to the problem, and it is often one of the most important factors in determining the success of a clustering approach.

### Common Distance Metrics 

There are many ways to define distance between data points, and for simplicity, we can group them into two broad categories. This classification helps organize the wide variety of distance measures, even though the boundaries between categories are not always strict.

The first category includes **Euclidean and spherical geometry distances**, which rely on the spatial location of points. These methods assume that data points exist in a geometric space — either flat (Euclidean) or curved (spherical) — and compute distances based on their coordinates. Examples include standard Euclidean distance in Cartesian space or great-circle distance when working with points on the Earth’s surface.

The second category consists of **non-Euclidean and non-spherical distances**, which do not depend directly on geometric coordinates. Instead, they measure similarity or dissimilarity using other properties of the data, such as shared attributes, correlations, or distributions. These methods are especially useful when the data cannot be naturally embedded in a geometric space or when alternative notions of similarity are more meaningful.

It is important to note that this is a coarse and somewhat subjective categorization. In practice, many distance measures may not fit neatly into one group, and alternative classification schemes can be equally valid depending on the context.

#### Euclidean and Spherical Distances

Euclidean and spherical distance measures are among the most commonly used approaches for quantifying similarity when data points can be represented in a geometric space. These methods rely on the coordinates of points and compute distances based on their spatial relationships.

The **Euclidean distance** is the most widely used metric and represents the straight-line distance between two points in a flat (Cartesian) space. For two points $P$ and $Q$ with coordinates $(p_1, p_2)$ and $(q_1, q_2)$, respectively, the distance is given by:

$$
d_{PQ} = \sqrt{(p_1-q_1)^2 + (p_2-q_2)^2}
$$

This concept naturally extends to higher dimensions and is often the default choice when working with numerical feature vectors.

Another commonly used measure is the **Manhattan distance**, also known as the taxicab or city-block distance. Instead of measuring the straight-line path, it computes the distance as the sum of absolute differences along each dimension. This can be thought of as the distance one would travel along a grid of city streets, making it particularly useful in scenarios where movement is constrained to orthogonal directions.

In contrast, **geodesic distance** is used when data points lie on a curved surface, such as the Earth. Rather than measuring distance through the interior of the space, it computes the shortest path along the surface itself. This makes it especially relevant for applications involving geographic coordinates, where straight-line (Euclidean) distance would not accurately reflect real-world distances.

<center><img src="https://www.dropbox.com/scl/fi/p5w2tekblsdgcan5tpwyg/distances.png?rlkey=wugrkrg7ptattnv0cjwrofcjz&st=xzwzym7w&dl=1" alt="drawing" width="450"/></center><br>

Together, these distance measures illustrate how geometric assumptions about the data influence how similarity is computed and interpreted.

#### Non-Euclidean Distance: Edit Distance

Edit distance is a commonly used non-Euclidean distance measure for comparing strings. It is defined as the minimum number of single-character operations — insertions, deletions, or substitutions — required to transform one string into another. Unlike geometric distances, edit distance does not rely on spatial coordinates but instead measures how structurally different two sequences are.

For example, consider the strings `x=AGACGTAG` and `y=GTTCAGA`. To convert `x` into `y`, we can perform a sequence of operations such as deleting characters, substituting one character for another, or inserting new characters. Each operation contributes a cost of one, and the goal is to find the sequence of operations with the smallest total cost.

<center><img src="https://www.dropbox.com/scl/fi/x3etqawadd8bu8dq8sikj/edit_distance.png?rlkey=fi5tz4i21284bqzymkwf5gwuy&st=9luzzxay&dl=1" alt="drawing" width="600"/></center><br>

In this example, transforming `x` into `y` requires six such operations, so the edit distance between the two strings is 6. This type of distance is widely used in applications such as spell checking, DNA sequence analysis, and text processing, where similarity is better captured through structural transformations rather than geometric proximity.

#### Non-Euclidean Distance: Jaccard Distance

Jaccard distance is a commonly used similarity-based metric for comparing sets or binary attributes. It measures how different two sets are by comparing their overlap relative to their combined size. First, the Jaccard index quantifies similarity as the proportion of shared elements between two sets out of all unique elements present in either set. The corresponding distance is then defined as one minus this similarity.

Formally, the Jaccard distance is given by:

$$ d_{jaccard} = 1 - \frac{|x \cap y|}{|x \cup y|} $$

This makes it particularly useful in settings where data can be represented as sets, such as collections of symptoms, keywords, or user-item interactions.

For example, consider two patients described by the presence (1) or absence (0) of certain symptoms:

|   | Muscle Cramps | Mouth Sores | Easy Bruising | Eye Twitch | Metallic Taste |
|---|:-:|:-:|:-:|:-:|:-:|
| Patient_x | 1 | 1 | 0 | 1 | 0 |
| Patient_y | 1 | 0 | 0 | 1 | 1 |

<br>
The set of symptoms for Patient_x is {Muscle Cramps, Mouth Sores, Eye Twitch}, and for Patient_y it is {Muscle Cramps, Eye Twitch, Metallic Taste}. The intersection contains 2 shared symptoms, while the union contains 4 unique symptoms in total. Therefore, the Jaccard index is:

$$ \text{Jaccard\_index(Patient\_x, Patient\_y)}=\frac{\text{|\{Muscle Cramps, Eye Twitch\}|}}{\text{|\{Muscle Cramps, Mouth Sores, Eye Twitch, Metallic Taste\}|}} = \frac{2}{4}=0.5 $$

The corresponding Jaccard distance is:

$$ d_{jaccard} = 1 - \text{Jaccard\_index} = 1 - 0.5 = 0.5 $$

This measure is especially useful when the presence or absence of features is more meaningful than their magnitude, making it widely applicable in fields such as text mining, recommendation systems, and medical diagnosis.

In [ ]:
def generate_random_string(length, n):
    return [''.join(random.choices(string.ascii_letters +
                                   string.digits, k=length)) for _ in range(n)]


symptoms = ["Muscle Cramps", "Weight Gain", "Easy Bruising", "Metallic Taste", "Paranoia",
            "Leg Pain", "Gas and Bloating", "Mouth Sores", "Nausea, Upset Stomach", 
            "Rectal Bleeding", "Shortness of Breath", "Muscle Cramps", "Urine Odor",
            "Swollen Ankles and Feet", "Joint Cracking", "Eye Twitch", "Dry Skin"]

patient_ids = generate_random_string(6, 20)
print(patient_ids)

In [ ]:
patient_outcomes = np.random.choice([0, 1], size=(len(symptoms)))            
patient_outcomes

In [ ]:
data = [np.random.choice([0, 1], size=(len(symptoms))) for _ in range(len(patient_ids))]
data

In [ ]:
patients_data = pd.DataFrame(columns=symptoms, data=data, index=patient_ids)
patients_data.head()

In [ ]:
patients_data.shape

In [ ]:
def compute_Jaccard_distance(vec_1, vec_2):
    
    # All the symptoms shared by both
    intersection = sum((vec_1 == vec_2) & (vec_1 == 1))
    
    # All the symptoms that either one has
    union = sum((vec_1 == 1) | (vec_2 == 1))
    
    return 1 - intersection / union

In [ ]:
vec_1 = np.array([1, 0, 1, 1, 1])
vec_2 = np.array([1, 0, 0, 1, 1])

print(compute_Jaccard_distance(vec_1, vec_2))

### Distance Matrix

In many clustering and machine learning applications, it is useful to store all pairwise distances between data points in a structured form called a **distance matrix $M$**. This matrix provides a complete representation of how every point in the dataset relates to every other point in terms of the chosen distance metric.

The distance matrix has dimensions $n \times n$, where $n$ is the number of data points in the dataset. Each entry in the matrix corresponds to the distance between a pair of points.

For any two data points $p_i$ and $p_j$, the matrix is defined such that:

$$ M[i, j] = d(p_i, p_j)$$
$$ M[j, i] = d(p_j, p_i)$$

where $d(p_i, p_j)$ denotes the distance between points $p_i$ and $p_j$. In most standard settings, the distance function is symmetric, meaning $d(p_i, p_j) = d(p_j, p_i)$. As a result, the distance matrix $M$ is also symmetric.

This representation is particularly useful because it allows clustering algorithms to operate directly on precomputed relationships between points, rather than repeatedly recalculating distances during execution.

In [ ]:
# Compute all pair-wise Jaccard coefficients
dist_patients = np.zeros([len(patient_ids), len(patient_ids)])

for i in range(patients_data.shape[0]):
    for j in range(i, patients_data.shape[0]):
        vec_1 = patients_data.iloc[i]
        vec_2 = patients_data.iloc[j]

        # compute distance and round to two decimal points
        dist_patients[i, j] = round(compute_Jaccard_distance(vec_1, vec_2), 2)
        dist_patients[j, i] = dist_patients[i, j]
        
dist_patients.shape

In [ ]:
dist_patients

In [ ]:
from numpy import unravel_index

max_idx = unravel_index(np.argmax(dist_patients), dist_patients.shape)

print(np.nanmax(dist_patients), np.nanargmax(dist_patients), *max_idx)

patients_data.iloc[[max_idx[0], max_idx[1]]]

In [ ]:
dist_patients_temp = dist_patients.copy()

for i in range(patients_data.shape[0]):
    dist_patients_temp[i, i] = 1000

min_idx = unravel_index(np.argmin(dist_patients_temp), dist_patients_temp.shape)

print(np.nanmin(dist_patients_temp), np.nanargmin(dist_patients_temp), *min_idx)

patients_data.iloc[[min_idx[0], min_idx[1]]]

###  Devising Your Own Distance

When designing a distance (or similarity) function, there are certain mathematical properties that are typically desired. In the strict mathematical sense, a valid distance function — also called a metric—should satisfy three key properties.

First, it must satisfy **non-negativity**, meaning: 
$$d(x, y) \ge 0.$$

Second, it should satisfy **symmetry**, so that the distance is the same in both directions:
$$d(x, y) = d(y, x).$$

Third, it must satisfy the **triangle inequality**, which ensures consistency of indirect paths:
$$d(x, y) + d(y, z) \ge d(x, z).$$ 

Together, these properties ensure that the distance behaves in a way that is consistent with our geometric intuition.

In practice, however, it is often difficult to design a distance function that satisfies all of these conditions. Some useful measures may violate symmetry. For example, consider distances defined along directed paths, such as the shortest clockwise route between two points on a circular track. If movement is restricted to only clockwise directions (as in certain navigation problems or one-way road systems), then the distance from $p$ to $q$ may differ from the distance from $q$ to $p$.

Despite this, symmetry is usually a highly desirable property in clustering applications, since most algorithms assume that relationships between points are bidirectional. When a naturally asymmetric distance is available, it can often be converted into a symmetric one. A common approach is to average the two directional distances:

$$
    dS(x,y)= \frac{d(x,y)+d(y,x)}{2}
$$

This simple transformation produces a symmetric distance function that is often more suitable for standard clustering methods, while still preserving the structure captured by the original measure.

## Clustering Approaches

Clustering methods can be broadly grouped into a few intuitive strategies, each offering a different way to organize data points based on similarity. One of the most foundational approaches is hierarchical clustering, specifically the agglomerative variant. In this method, every individual point initially forms its own cluster. The algorithm then repeatedly identifies the two closest clusters and merges them into a single cluster. This merging process continues step by step, gradually building larger and larger clusters, until ultimately all points are combined into one overarching cluster. The result is often visualized as a tree-like structure, revealing relationships at multiple levels of granularity.

Another widely used approach is point-to-cluster assignment, where the number of clusters, denoted as k, is defined in advance. The algorithm begins by maintaining these k clusters and assigning each data point to the cluster it is closest to, typically based on some distance metric. This assignment is not done just once; instead, the process is repeated iteratively. After each round of assignments, cluster centers may be updated, and points are reassigned until the configuration stabilizes and no longer changes significantly. This iterative refinement makes the method both practical and efficient for many real-world datasets.

Beyond these two core strategies, there exists a wide spectrum of more advanced and specialized clustering techniques. These methods can be difficult to categorize neatly because they often blend ideas from multiple approaches or introduce entirely new frameworks. In recent years, probabilistic methods have gained popularity, offering a way to model uncertainty and variability within the data. Such approaches can provide more flexible and expressive clustering results, especially in complex or noisy environments.

### Agglomerative Hierarchical Clustering

<center><img src="https://www.dropbox.com/scl/fi/bmypbj1n7k3qfer95y1d0/hierarchical_clustering.png?rlkey=4akhyjgvexl6h72njzey8pf5a&st=srpbjxzw&dl=1" alt="drawing" width="650"/>


#### Dendrogram Representation

<center><img src="https://www.dropbox.com/scl/fi/s8cmq8qgzcr95mlh1fevo/hierarchical_clustering_dendo.png?rlkey=ll01031m52lm2dqs04czl6448&st=tgfwkj01&dl=1" alt="drawing" width="750"/>



In agglomerative hierarchical clustering, the central challenge lies in determining which pair of clusters should be merged at each step. While distances between individual data points are typically well-defined, extending this notion to distances between clusters is less straightforward. A common workaround is to represent each cluster by a single summary point, such as its centroid — the average of all points in the cluster. By computing the centroids for all clusters, we can then measure distances between these representative points and merge the pair whose centroids are closest. This approach works well when the data lives in a space where averaging is meaningful, such as Euclidean space.

However, the centroid-based approach does not generalize to all types of data. For example, if the data consists of DNA sequences, there is no obvious way to compute an “average” sequence in the same sense as averaging numerical vectors. DNA sequences are symbolic rather than numeric, and operations like averaging do not naturally apply. This limitation highlights an important issue: clustering methods that rely on centroids assume a structure in the data that may not exist in more complex or non-Euclidean settings.

To address this, hierarchical clustering can instead rely on alternative definitions of distance between clusters that do not require computing a centroid. These methods define inter-cluster distance based on pairwise distances between points in different clusters. For instance, **single linkage** (or minimum linkage) defines the distance between two clusters as the smallest distance between any pair of points across the clusters, which can lead to “chaining” effects. **Complete linkage** (or maximum linkage) uses the largest such distance, producing more compact clusters. **Average linkage** takes the mean of all pairwise distances between points in the two clusters, offering a balance between the extremes. Another widely used method is **Ward’s approach**, which merges clusters in a way that minimizes the increase in within-cluster variance at each step. These alternatives make hierarchical clustering flexible enough to handle a wide variety of data types and distance measures.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering

centers = [[-1, -1], [0, 0], [1, 1], [2, -1], [-2, 1]]
cluster_std = [0.1, 0.5, 0.2, 0.6, 0.15]

X, y = make_blobs(n_samples=150, centers=centers,
                  cluster_std=cluster_std, random_state=42)

plt.figure(figsize=(15, 4))

for idx, linkage in enumerate(['single', 'complete', 'average', 'ward']):
    agg = AgglomerativeClustering(n_clusters=5, linkage=linkage)
    labels = agg.fit_predict(X)
    print(f"{linkage:8}: {np.bincount(labels)}")

    # plot the cluster assignments
    plt.subplot(1, 4, idx+1)
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=40, cmap='coolwarm', alpha=0.7)
    plt.subplots_adjust(wspace = 0.1, hspace=0.4)
    plt.xlim(X[:,0].min()-0.5, X[:,0].max()+0.5)
    plt.ylim(X[:,1].min()-0.5, X[:,1].max()+0.5)
    plt.xticks([]), plt.yticks([])
    plt.title(f"linkage = {linkage}", fontsize=20)

plt.tight_layout()


Hierarchical clustering offers several appealing advantages, particularly in terms of simplicity and interpretability. The algorithm is conceptually straightforward: it repeatedly merges clusters based on a notion of closeness, making it easy to implement and explain. One of its biggest strengths is the resulting tree-like structure, or dendrogram, which provides a clear visualization of how clusters are formed over time. This visualization allows you to explore the data at multiple levels of granularity, making hierarchical clustering a powerful tool for summarizing relationships and gaining a holistic view of the dataset.

At the same time, this approach comes with notable drawbacks. Hierarchical clustering can be computationally expensive, especially for large datasets, since it requires calculating and updating distances between many pairs of points or clusters. As the dataset grows, the cost can quickly become prohibitive. Additionally, while the dendrogram is informative, extracting meaningful clusters from it is not always straightforward.

A key challenge lies in deciding where to “cut” the tree to form clusters. In the dendrogram, individual data points appear as leaves, and the distances between them are reflected in the height at which they are merged. Ideally, large vertical gaps in the tree would suggest natural divisions between clusters. However, in many real-world datasets, such clear separations do not exist. As a result, different choices of cut height can lead to very different clustering outcomes, making the final interpretation somewhat subjective and dependent on the analyst’s judgment.

### Assigning Points to Clusters: K-Means Clustering

K-means clustering is one of the most widely used methods for partitioning data into a predefined number of groups. The process begins by choosing the number of clusters, denoted as K, which determines how many groupings the algorithm will attempt to find. From there, the algorithm initializes K representative points — called centroids — often by selecting random points in the data space. These centroids act as the initial “centers” of the clusters.

<center><img src="https://www.dropbox.com/scl/fi/zh60t33h9dwu3veokukvk/kmeans_ab.png?rlkey=b9b4z1y08iu32o54brpvv39gu&st=cf5q4ox4&dl=1" width="500"/></center>

Once initialized, the algorithm alternates between two key steps. First, each data point is assigned to the cluster whose centroid is closest, typically using Euclidean distance. You can think of this step as “coloring” each point according to its nearest centroid. After all points have been assigned, the second step recomputes the centroids by taking the average of all points within each cluster. These updated centroids better reflect the current grouping of the data.

<center><img src="https://www.dropbox.com/scl/fi/sga6j5kq76az7kc18ixnn/kmeans_cd.png?rlkey=khrkup0pr23x726z2b254tayg&st=t2iep39f&dl=1" width="500"/></center>

This assignment-and-update cycle repeats iteratively. With each iteration, points may switch clusters as centroids shift to more representative positions. Eventually, the process converges when the assignments stop changing and the centroids stabilize. At this point, the clustering is considered complete.

It is important to note that k-means assumes the data exists in a Euclidean space where calculating averages is meaningful. Because of this, it does not naturally handle categorical or non-numeric data. Despite this limitation, its simplicity, efficiency, and effectiveness on many real-world datasets make it a cornerstone algorithm in clustering.

In [ ]:
# code for plotting clustering result

def plot_data(X):
    plt.plot(X[:, 0], X[:, 1], 'k.', markersize=2, alpha=0.3)

def plot_centroids(centroids, weights=None, circle_color='y', cross_color='r'):
    if weights is not None:
        centroids = centroids[weights > weights.max() / 10]
        
    plt.scatter(centroids[:, 0], centroids[:, 1],
                marker='x', s=4, linewidths=10,
                color=cross_color, zorder=11, alpha=1)

def plot_decision_boundaries(clusterer, X, resolution=1000, show_centroids=True,
                             show_xlabels=True, show_ylabels=True):
    mins = X.min(axis=0) - 0.1
    maxs = X.max(axis=0) + 0.1
    xx, yy = np.meshgrid(np.linspace(mins[0], maxs[0], resolution),
                         np.linspace(mins[1], maxs[1], resolution))
    Z = clusterer.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contour(Z, extent=(mins[0], maxs[0], mins[1], maxs[1]), linewidths=1, colors='b')
    plot_data(X)
    
    if show_centroids:
        plot_centroids(clusterer.cluster_centers_)

    if show_xlabels:
        plt.xlabel("$x_1$", fontsize=14)
    else:
        plt.tick_params(labelbottom=True)
        
    if show_ylabels:
        plt.ylabel("$x_2$", fontsize=14, rotation=0)
    else:
        plt.tick_params(labelleft=True)

In [ ]:
# data generation

from sklearn.datasets import make_blobs

blob_centers = np.array(
    [[ 0.0,  2.3],
     [-1.5,  2.3],
     [-2.0,  1.5],
     [-2.8,  2.8]])
blob_std = np.array([0.4, 0.3, 0.1, 0.6])

X, y = make_blobs(n_samples=2000, centers=blob_centers, 
                  cluster_std=blob_std, random_state=0)
plt.scatter(X[:, 0], X[:, 1], s=1);

In [ ]:
# K-means clustering using scikit-learn

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, n_init=10)
kmeans.fit_predict(X)

print(kmeans.cluster_centers_)
print(kmeans.labels_)

In [ ]:
print(kmeans.inertia_)
print(kmeans.score(X))

In [ ]:
plt.figure(figsize=(6, 3.5))
plot_decision_boundaries(kmeans, X, show_xlabels=False, show_ylabels=False)
plt.title("K = 4", fontsize=12);

In [ ]:
kmeans_per_k = [KMeans(n_clusters=k, n_init=10).fit(X) for k in range(2, 9)]
inertias = [model.inertia_ for model in kmeans_per_k]

plt.figure(figsize=(6, 3.5))

plt.plot(range(2, 9), inertias, "bo-")
plt.xlabel("K", fontsize=12)
plt.ylabel("Inertia", fontsize=12);

#### k-Means Clustering Results

K-means clustering is widely appreciated for its practical performance, but its results come with a mix of strengths and limitations. One important drawback is that the algorithm is non-deterministic. Because it typically begins with randomly chosen initial centroids, different runs can produce different cluster assignments. This sensitivity to initialization means it is common practice to run the algorithm multiple times and either select the best result based on some criterion or form a consensus across runs.

Another limitation stems from its reliance on Euclidean space. Since k-means computes centroids as averages, it requires numerical data where such operations are meaningful. As a result, it cannot be directly applied to categorical or non-Euclidean data, where defining an “average” is difficult or even impossible. In these cases, alternative approaches are needed.

One such alternative is k-medoids clustering. Instead of representing clusters by centroids (which may not correspond to actual data points), k-medoids selects representative points — called medoids — that minimize the average distance to all other points in the cluster. Because medoids are actual observations, this method is more robust to outliers and tends to produce more stable clusterings, especially in noisy datasets.

In terms of performance, k-means is generally efficient and converges relatively quickly, often within 10 to 50 iterations. However, poor initializations can occasionally slow convergence, and in rare cases, the algorithm may take longer to stabilize. Despite these caveats, k-means remains a highly effective and commonly used clustering technique, performing well across a wide range of real-world applications.

## How Do We Evaluate the Results of Clustering

Evaluating clustering results can be challenging because, unlike supervised learning, there is often no ground truth to compare against. Instead, we rely on intuitive principles that define what “good” clusters should look like. At a high level, a strong clustering solution should group together points that are very similar to one another while keeping dissimilar points in separate clusters. These two ideas are captured by the concepts of **cohesion** and **separation**.

Cohesion refers to how closely related the points within the same cluster are. Ideally, points in a cluster should be tightly grouped, meaning the average distance between them is small. This indicates that the cluster represents a meaningful grouping of similar observations. On the other hand, separation measures how distinct different clusters are from one another. Good clustering ensures that points in different clusters are far apart, reflecting clear boundaries between groups.

One simple and effective way to combine these ideas into a single evaluation criterion is by taking the **ratio of separation to cohesion**. A good clustering solution will have high separation (clusters are far apart) and high cohesion (points within clusters are close together, i.e., small within-cluster distances), resulting in a large value for this ratio. In other words, the larger the separation relative to cohesion, the better the clustering quality. This type of metric helps quantify what is otherwise a qualitative judgment, making it easier to compare different clustering results objectively.

### The Silhouette Coefficient

The **silhouette coefficient** provides a practical way to quantify the balance between separation and cohesion for a clustering result. It serves as a per-point measure that captures how well each data point fits within its assigned cluster compared to other clusters. By averaging these values across all points, we obtain the **silhouette score**, which summarizes the overall quality of the clustering.

For each point $i$, two quantities are computed. The first, $a_i$, is the average distance between point $i$ and all other points in the same cluster. This reflects cohesion, so smaller values indicate that the point is well matched to its own cluster. The second, $b_i$, is the average distance between point $i$ and points in other clusters (typically taken as the minimum average distance to another cluster). This captures separation, where larger values indicate that the point is well separated from neighboring clusters.

The silhouette coefficient for point $i$ is defined as

$$
S_i = \frac{b_i - a_i}{max(a_i, b_i)}
$$

This value ranges from −1 to 1 and provides an interpretable measure of clustering quality at the point level. Values close to 1 indicate that $b_i$ is much larger than $a_i$, meaning the point is tightly grouped within its cluster and far from others — an ideal situation. Values near 0 suggest that the point lies near a boundary between clusters. Negative values indicate that $a_i$ exceeds $b_i$, meaning the point is closer, on average, to another cluster than its own, which signals poor clustering or potential misassignment.

By examining both individual silhouette coefficients and their average, we gain a clearer, quantitative sense of how well the clustering achieves the desired balance of high cohesion and high separation.

In [ ]:
### Use minimum distance

from scipy.spatial import distance

def compute_b(pt_cl1, c2):
    min_dist = np.inf
    
    for j, pt2 in enumerate(c2):
        d = distance.euclidean(pt_cl1, pt2)
        if d < min_dist:
            min_dist = d
            
    return min_dist

In [ ]:
def compute_a(cluster, target_pt_id):
    distances = []
    
    for other_pt_id in range(len(cluster)):
        if target_pt_id != other_pt_id:
            distances.append(distance.euclidean(cluster[target_pt_id], 
                                                cluster[other_pt_id]))
            
    return np.mean(distances)

In [ ]:
def pt_silhouette(cluster, target_pt_id, other_cluster):
    a_i = compute_a(cluster, target_pt_id)
    b_i = compute_b(cluster[target_pt_id], other_cluster)
    
    return (b_i - a_i) / max(a_i, b_i)

### Computing Cluster-Wide Silhouette Coefficient

The silhouette coefficient can also be used to assess clustering quality at the level of individual clusters. In this case, we compute the average silhouette coefficient for all data points within a given cluster. This cluster-level average provides a useful summary of how well that specific cluster is formed. Higher values indicate that points in the cluster are, on average, well matched to their own cluster (high cohesion) and clearly separated from other clusters (high separation), while lower values suggest weaker structure or potential overlap with other clusters.

Beyond individual clusters, we can also evaluate the clustering solution as a whole by averaging silhouette coefficients over all data points in the dataset. This global average, often referred to as the overall **silhouette score**, provides a concise estimate of the quality of the entire clustering. A higher overall score indicates that, on average, points are both well grouped within their clusters and well separated from other clusters, making it a widely used summary metric for comparing different clustering results.

In [ ]:
# For simplicity, the solution assumes two clusters 
# Need to consider additional clusters in solution

def cluster_silhouette(cluster, other_cluster):
    silhouette_coeffs = []
    for pt_id in range(len(cluster)):
        silhouette_coeffs.append(pt_silhouette(cluster, pt_id, other_cluster))

    return(np.mean(silhouette_coeffs))

In [ ]:
mean_c1 = [8, 12]
cov_c1 = [[1, 0], [0, 1]]
c1 = np.random.multivariate_normal(mean_c1, cov_c1, 40)

mean_c2 = [4, 4]
cov_c2 = [[1, 0], [0, 1]]
c2 = np.random.multivariate_normal(mean_c2, cov_c2, 29)

plt.figure(figsize=(4, 4))
plt.scatter(c1[:, 0], c1[:, 1])
plt.scatter(c2[:, 0], c2[:, 1])

plt.xticks([], [])
plt.yticks([], []);

In [ ]:
avg_sil_coef_c1 = cluster_silhouette(c1, c2)
avg_sil_coef_c2 = cluster_silhouette(c2, c1)

print(f"{avg_sil_coef_c1:.4f},  {avg_sil_coef_c2:.4f}")
print(f"Overall score: {np.mean([avg_sil_coef_c1, avg_sil_coef_c2]):.4f}")

In [ ]:
mean_c1 = [8, 12]
cov_c1 = [[6, 0], [0, 9]]
c1 = np.random.multivariate_normal(mean_c1, cov_c1, 40)

mean_c2 = [4, 4]
cov_c2 = [[2, 0], [0, 2]]
c2 = np.random.multivariate_normal(mean_c2, cov_c2, 29)

plt.figure(figsize=(4, 4))
plt.scatter(c1[:, 0], c1[:, 1])
plt.scatter(c2[:, 0], c2[:, 1])

plt.xticks([], [])
plt.yticks([], []);

In [ ]:
avg_sil_coef_c1 = cluster_silhouette(c1, c2)
avg_sil_coef_c2 = cluster_silhouette(c2, c1)

print(f"{avg_sil_coef_c1:.4f},  {avg_sil_coef_c2:.4f}")
print(f"Overall score: {np.mean([avg_sil_coef_c1, avg_sil_coef_c2]):.4f}")

In [ ]:
mean_c1 = [8, 12]
cov_c1 = [[8, 0], [0, 15]]
c1 = np.random.multivariate_normal(mean_c1, cov_c1, 40)

mean_c2 = [4, 4]
cov_c2 = [[4, 0], [0, 10]]
c2 = np.random.multivariate_normal(mean_c2, cov_c2, 29)

plt.figure(figsize=(4, 4))
plt.scatter(c1[:, 0], c1[:, 1])
plt.scatter(c2[:, 0], c2[:, 1])

plt.xticks([], [])
plt.yticks([], []);

In [ ]:
avg_sil_coef_c1 = cluster_silhouette(c1, c2)
avg_sil_coef_c2 = cluster_silhouette(c2, c1)

print(f"{avg_sil_coef_c1:.4f},  {avg_sil_coef_c2:.4f}")
print(f"Overall score: {np.mean([avg_sil_coef_c1, avg_sil_coef_c2]):.4f}")

### Using Silhouettes to Decide on Best Number of Cluster 

The silhouette score can also be used as a practical tool for selecting the number of clusters in a dataset. Since clustering algorithms like k-means require the number of clusters k to be specified in advance, we can evaluate different choices of k by running the algorithm multiple times and computing the overall silhouette coefficient for each value.

In practice, we run the k-means algorithm for a range of candidate values of k, and for each one compute the average silhouette coefficient across all data points. The idea is that the “best” number of clusters should produce the most natural separation in the data, which will be reflected as a peak in the silhouette score. Thus, the value of k that maximizes the overall silhouette score is often chosen as the most appropriate number of clusters.

Note: We need to run the k-means algorithm several times for each value of k because the algorithm is sensitive to its initialization. Since the initial centroids are typically chosen at random, different starting points can lead to different final cluster assignments and potentially different local optima. As a result, a single run may not reflect the best possible clustering for that value of k. By running k-means multiple times and comparing results, we reduce the impact of poor initialization and obtain a more reliable estimate of clustering quality.

In [ ]:
from sklearn.metrics import silhouette_score

s_scores = [silhouette_score(X, model.labels_) for model in kmeans_per_k]

plt.figure(figsize=(8, 3))
plt.plot(range(2, 9), s_scores, "bo-")
plt.xlabel("K", fontsize=14)
plt.ylabel("Silhouette score", fontsize=12)
plt.show()